# 02. 네 도구와 실행 모드 / 60분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

시뮬레이터, 과제, 모델 연결, 정책의 역할을 분리하면 오류가 난 위치를 찾을 수 있습니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1yAFoaiH15_qTveoSi5r1gF1TPE7bFf2CrLVLUolsg4k/edit#slide=id.d5_lab02_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![네 도구와 실행 모드의 입력·처리·출력](../교재그림/guided-vla/day5-02.svg)

inspect는 reset과 저장까지만 진행합니다. infer는 정책 예측까지 추가하지만 행동을 적용하지 않습니다. rollout은 env.step과 재관측까지 반복합니다. 실행 추적 예제는 이 분기만 보여 주며 실제 물리·모델 실행은 04부터 합니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![5일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day5-system.svg)

브라우저는 실행을 요청하고 결과를 표시합니다. 커널이 호출한 별도 runner가 물리 환경과 정책을 연결합니다. 01·02의 합성 셀은 구조 설명용이고 실제 환경 실행은 04, 모델 실행은 06부터입니다. 아래 단원별 도식에서 이번에 실제로 실행할 범위를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 네 도구의 책임

- MuJoCo: 물리 상태 변화와 렌더링
- LIBERO: 과제·초기 배치·성공 조건
- LeRobot: 환경·정책·전후처리 연결
- SmolVLA: 입력에서 행동 묶음을 예측하는 정책

라이브러리를 이름만 외우면 오류가 발생했을 때 어디를 볼지 알기 어렵습니다. MuJoCo는 물리 시뮬레이션과 장면 렌더링을 제공하고, LIBERO는 어떤 물체를 어떻게 다뤄야 하는지 과제와 초기상태를 제공합니다. 수업 코드는 LeRobot의 LIBERO 환경 어댑터와 SmolVLA 정책 및 processor를 사용해 둘을 연결합니다.

SmolVLA는 성공 판정기를 대체하지 않습니다. 정책이 낸 행동을 환경에 적용하고 info의 is_success를 확인해야 성공 여부를 알 수 있습니다. 반대로 물리 엔진은 언어 지시를 이해해 행동을 계획하는 모델이 아닙니다. 계획과 물리 적용, 평가 책임이 분리되어 있습니다.

노트북은 이러한 구성 요소를 호출하는 진입점입니다. 무거운 실행은 별도 Python 프로세스의 run_episode.py가 맡고, 노트북은 결과 파일을 읽어 표와 그래프를 표시합니다. 브라우저 화면, Jupyter 커널, 실행 프로세스를 하나로 보지 않는 것이 중요합니다.


### 2. 실행 아키텍처: 세 모드의 공통점과 분기

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day5-02.svg)

- 모든 모드: make_env → reset → 초기 관측 저장
- inspect: 정책을 로드하지 않고 여기서 종료
- infer: 정책 로드·행동 예측 후 저장; step 없음
- rollout: 행동 적용·재관측·성공 검사를 반복

inspect·infer·rollout은 서로 다른 세 모델이 아니라 같은 runner의 실행 범위입니다. run_episode.run은 출력 폴더와 incomplete 상태를 먼저 준비한 뒤 환경을 만들고 reset합니다. 초기 영상과 상태를 저장하는 공통 부분이 끝난 뒤 mode에 따라 분기합니다.

inspect에서 성공적으로 결과가 나왔다면 환경 생성·초기상태·영상 저장 경로를 확인한 것입니다. 모델 적재나 GPU 추론은 아직 확인하지 않았습니다. infer가 완료되면 정책과 전후처리 및 출력 형상을 확인하지만 행동은 적용하지 않습니다. rollout에서만 env.step이 호출되어 상태가 바뀌고 성공 판정을 읽습니다.

이 순서는 오류를 좁히는 실행 전략이기도 합니다. inspect가 실패하는데 rollout부터 반복해도 모델 문제인지 환경 문제인지 분리하기 어렵습니다. 02의 합성 호출 추적은 이 분기 구조만 출력하고, 04부터 실제 환경과 파일을 사용합니다.


### 3. 코드 추적: 공통 준비와 finally

- 출력 경로 보호 → 환경 reset → 초기 파일 저장
- mode 분기 → 필요할 때만 정책 로드
- rollout 내부에서 step·관측·종료 조건 갱신
- finally에서 영상 닫기·result.json·GUI·환경 정리

try는 정상 실행 경로, except는 오류 상태를 기록하는 경로, finally는 정상 반환이나 예외에도 수행되는 정리 경로입니다. inspect에서 return해도 finally가 실행되어 result.json과 GUI 마무리가 처리됩니다. 코드 아래쪽이라는 이유로 정상 실행에서 도달하지 않는 부분이라고 읽으면 안 됩니다.

result는 처음 incomplete로 시작합니다. 정상 범위가 끝나면 completed로 바뀌고 예외가 발생하면 error와 함께 incomplete가 보존됩니다. 파일 저장이나 환경 정리 오류도 결과 상태에 영향을 줄 수 있습니다. 결과 파일의 존재와 status 값을 함께 보는 이유입니다.

모드 분기를 수정할 때 infer에 env.step을 추가하면 더 이상 예측만 보는 실습이 아닙니다. inspect에 정책 로드를 앞당기면 환경 확인에도 모델·GPU가 필수가 됩니다. 함수 순서의 작은 변경이 각 모드가 보장하는 검증 범위를 바꿀 수 있습니다.


### 4. 화면 정지와 실제 실행 상태

- MODEL LOADING·INFERENCE 동안 환경은 대기합니다.
- ACTION EXECUTION에서 env.step이 적용됩니다.
- FINISHED 뒤 창을 닫아야 노트북 셀이 반환됩니다.
- 영상 fps와 실제 벽시계 처리 속도는 다릅니다.

noVNC의 MuJoCo 화면이 잠시 멈춰 있어도 모델을 적재하거나 추론하는 단계라면 정상입니다. 이 runner는 비동기로 물리를 계속 진행하면서 별도 정책을 받는 구조가 아니라 정책 예측 뒤 행동을 적용하는 동기 흐름입니다. 화면의 단계 문구와 steps, Inferences를 같이 봅니다.

LiveViewer는 원본 물리 상태를 복사한 관찰용 모델·데이터를 표시합니다. 관찰 창의 카메라를 움직이는 일과 정책 입력 카메라를 바꾸는 일은 다릅니다. GUI에서 장면을 조작해도 원본 과제 상태를 수정하는 인터페이스로 설계되지 않았습니다.

rollout.mp4는 fps=20으로 저장됩니다. 이는 재생 시간의 설정이며 모델 추론을 포함한 실제 벽시계 시간의 기록이 아닙니다. 화면이 부드럽게 재생돼도 추론이 실시간으로 수행되었다고 결론 내리지 않습니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [scripts/run_episode.py](scripts/run_episode.py) |
| 바꾸어 볼 조건 | `설명용 mode=inspect / infer / rollout` |
| 예상되는 차이 | 세 모드의 호출 순서를 출력하고 env.step과 재관측이 rollout에서만 나타나는 것을 봅니다. |
| 직접 구현·수정할 때 | 행동 묶음마다 reset하면 과제가 계속 처음으로 돌아갑니다. 추론 대기와 행동 적용을 구분하고 영상 fps를 벽시계 제어 주파수로 해석하지 않습니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


## 1. “그릇을 집어 접시에 놓기”에는 무엇이 필요할까요?

탁자에 로봇팔·그릇·접시가 있다고 생각해 봅시다. 아래는 역할을 설명하기 위한 예시이며, 실제 실행할 과제의 지시는 이후 실습에서 확인합니다.

로봇이 그릇으로 다가가려면 먼저 그릇과 그리퍼의 현재 모습을 볼 수 있어야 합니다. 그리퍼를 움직이면 로봇의 자세가 바뀌고, 그릇에 닿으면 접촉 결과도 달라집니다. 접시 위에 놓았을 때에는 목표를 달성했는지 확인해야 합니다.

| 필요한 것 | 없다면 생기는 문제 | 이번 수업에서 맡는 구성요소 |
|---|---|---|
| 움직임·접촉을 계산할 가상 세계와 카메라 | 행동 숫자가 나와도 실제 변화나 다음 영상을 만들 수 없음 | **MuJoCo** |
| 과제·장면·초기 배치·성공 조건 | 무엇을 해야 하고 언제 성공인지 정할 수 없음 | **LIBERO** |
| 환경과 정책의 데이터 형식을 연결하는 도구 | 영상·상태·행동 형식이 달라 서로 바로 사용할 수 없음 | **LeRobot과 교재 연결 코드** |
| 관측에서 다음 행동을 예측하는 정책 | 어떤 움직임을 할지 결정할 수 없음 | **SmolVLA** |

**같은 종류의 프로그램 네 개를 골라 쓰는 것이 아닙니다.** 역할이 다른 도구들을 한 실습 시스템으로 연결합니다. SmolVLA는 LeRobot에서 지원하는 정책 중 하나이며, LIBERO 환경은 MuJoCo 기반으로 동작합니다.


## 2. 전체 구성을 먼저 봅시다

![LIBERO가 과제 조건을 정의하고 MuJoCo가 환경을 계산하며 LeRobot과 연결 코드가 SmolVLA의 입력과 출력을 연결하는 구성도](../교재그림/vla-stack-roles.svg)

그림에서 **환경** 은 로봇과 물체가 놓인 장면, 현재 상태, 행동을 적용하고 새 관측을 받는 실행 대상을 뜻합니다. 환경과 정책은 별개의 역할입니다.

**교재 실행 코드** 는 각 구성요소를 호출하는 순서를 관리합니다. 환경을 초기화하고, 정책에 입력을 넣고, 행동 일부를 실행하고, 로그·영상을 저장합니다. 모든 일을 LeRobot이나 모델 하나가 자동으로 처리한다고 생각하면 오류가 났을 때 원인을 찾기 어렵습니다.

그림은 역할을 설명한 개념도입니다. 실제 구현에서는 robosuite 같은 중간 계층도 사용합니다. 이 단원에서는 네 주요 구성요소와 교재 실행 코드의 책임부터 구분합니다.


## 3. MuJoCo — 행동을 적용하면 무엇이 달라질까요?

MuJoCo는 **물리 시뮬레이션 엔진** 입니다. 로봇의 관절과 물체, 질량·마찰·접촉 등의 설정을 이용해 시간이 진행될 때 상태가 어떻게 바뀌는지 계산합니다. 가상 카메라에서 장면을 그려 영상으로 만드는 렌더링 기능도 제공합니다. [MuJoCo 공식 설명](https://mujoco.readthedocs.io/en/stable/)

예를 들어 로봇이 그릇을 밀면 그릇이 움직이거나 다른 물체와 부딪힐 수 있습니다. 그 결과로 다음 카메라 영상도 달라집니다. **그릇을 어디로 옮길지는 정책이 예측하고, 행동을 적용한 물리적 결과는 시뮬레이터가 계산합니다.**

가상 카메라는 장면 안에 배치한 카메라입니다. 이번 수업의 외부 시점과 손목 시점 영상은 USB 웹캠이 아니라 이 가상 장면에서 생성됩니다.

시뮬레이션은 설정된 모델과 수치 계산에 따른 근사입니다. 시뮬레이터에서 그릇을 잡았다고 실제 그리퍼의 마찰·유연성·카메라 조건에서도 같은 결과가 보장되지는 않습니다.


## 4. LIBERO — 무엇을 하고, 어디서 시작하고, 언제 성공할까요?

물리 엔진만으로는 “이번에는 어떤 물체를 어디로 옮겨야 한다”는 과제가 생기지 않습니다. LIBERO는 로봇 조작을 평가하기 위한 **과제와 환경 구성** 을 제공합니다. 과제의 언어 지시, 물체 배치와 초기 상태, 성공 조건을 함께 다룹니다. [LIBERO 공식 저장소](https://github.com/Lifelong-Robot-Learning/LIBERO)

| 구분 | 그릇 옮기기 예시에서의 의미 |
|---|---|
| 과제 | 어떤 그릇을 어느 목표로 옮길 것인가 |
| 초기 상태 | 로봇과 물체가 어떤 배치에서 시작할 것인가 |
| 성공 조건 | 목표를 달성했다고 판단할 상태 조건은 무엇인가 |

**“그릇 위치가 바뀐다”** 는 말에도 두 경우가 있습니다. 실험 시작 전에 다른 초기 배치를 선택하는 것은 과제 초기화의 문제입니다. 실행 중 로봇이 밀어서 위치가 바뀌는 것은 물리 계산의 결과입니다.

이번 수업의 `initial_state`는 준비된 초기 배치 선택이고, `seed`는 난수 설정입니다. **seed만 바꾸는 것을 초기 배치 번호를 바꾸는 것과 동일하게 해석하지 않습니다.**

성공 여부는 모델이 “성공했습니다”라고 말하는 것으로 정하지 않습니다. 실행 중 환경에서 전달되는 `is_success`를 교재 코드가 읽습니다. 영상도 함께 보되, 환경이 정한 성공 조건과 실제 화면이 무엇을 보여 주는지 구분합니다.


## 5. LeRobot과 연결 코드 — 왜 형식을 맞춰야 할까요?

환경에서 얻은 영상·상태와 정책이 기대하는 입력은 표현 방식이 다를 수 있습니다. 예를 들어 영상의 축 순서나 값 범위, 카메라 이름, 로봇 상태의 구성과 정규화 기준이 맞아야 합니다.

**전처리** 는 정책에 넣기 전에 입력을 정해진 형식으로 바꾸는 과정이고, **후처리** 는 정책 출력을 환경에서 사용할 행동 표현으로 복원하는 과정입니다. LeRobot은 정책과 연결되는 처리 파이프라인을 제공합니다. [LeRobot 처리 파이프라인](https://huggingface.co/docs/lerobot/main/en/api/processor)

| 연결 지점 | 이번 교재에서 확인할 것 | 틀리면 생길 수 있는 일 |
|---|---|---|
| 환경 → 입력 구성 | 영상 2개, 언어 지시, 상태 8개 | 필수 카메라·상태 누락 |
| 입력 → 정책 | 영상 방향·축·값 범위·키 이름·정규화 | 크기는 맞아도 다른 의미의 입력 전달 |
| 정책 → 실행 행동 | 후처리, 7개 행동 성분, 유한한 값·허용 범위 | 잘못된 단위·축으로 움직임 적용 |

여기에는 LeRobot의 정책·환경 어댑터뿐 아니라 [교재 연결 코드](scripts/course_runtime.py)도 참여합니다. 정책·환경마다 적용할 변환이 다르므로 숫자 개수만 맞추면 된다고 생각하면 안 됩니다.

“영상이 거꾸로 들어갔다”는 문제를 학습된 정책의 성능 문제로만 해석하지 마세요. **입력 연결이 맞는지 먼저 확인해야 모델의 판단을 평가할 수 있습니다.**


## 6. SmolVLA — 현재 상황에서 어떤 행동을 할까요?

SmolVLA는 영상·언어·로봇 상태를 이용해 **행동 묶음** 을 예측하는 학습된 정책입니다. 환경이나 성공 판정기가 아니라, 행동을 만드는 모델입니다. [SmolVLA 공식 설명](https://huggingface.co/blog/smolvla)

이번 수업에서는 준비된 LIBERO 작업용 체크포인트를 사용합니다. 정책 출력은 **50시점×7개 행동 값** 이고, 기준 실행은 앞 **10개** 를 적용한 뒤 다시 관측합니다. 이것은 현재 교재의 설정입니다.

정책은 로봇이 다음에 어떻게 움직일지 예측하지만, 그 행동이 실제로 그릇을 잡았는지는 환경을 실행해 봐야 알 수 있습니다. 접촉·물체 변화는 MuJoCo 기반 환경에서 계산하고, 과제의 성공 조건은 LIBERO 환경에서 확인합니다.

정책을 다른 것으로 바꾸더라도 입력·출력과 로봇의 동작 의미가 맞아야 합니다. “같은 크기의 배열을 반환한다”는 것만으로 교체 가능한 것은 아닙니다.


## 7. 한 번의 실행을 끝까지 따라가 봅시다

<!-- procedure:lesson-38-95b298:start -->
![한 번의 환경 실행을 따라가기](../교재그림/vla-stack-cycle.svg)

교재 실행기가 도구를 연결합니다. 행동 묶음마다 초기 배치로 reset하지 않습니다.
<!-- procedure:lesson-38-95b298:end -->


![과제 초기화부터 관측, 입력 변환, 행동 예측, 후처리, 행동 적용, 재관측과 성공 확인으로 이어지는 순서](../교재그림/vla-stack-cycle.svg)

| 순서 | 실제로 일어나는 일 | 관찰할 결과 |
|---|---|---|
| ① 과제와 초기 상태 선택 | LIBERO 과제를 만들고 환경을 초기화 | 지시, 로봇과 물체의 초기 배치 |
| ② 관측 얻기 | 환경에서 가상 영상과 로봇 상태를 받음 | 외부·손목 영상, 상태 값 |
| ③ 입력 연결 | 교재 코드와 전처리가 정책 입력을 구성 | 맞는 카메라·축·값 범위·상태 |
| ④ 행동 예측 | SmolVLA가 현재 입력으로 행동 묶음 생성 | 50×7 행동 값 |
| ⑤ 후처리와 일부 실행 | 행동을 복원·검사하고 앞 10개를 순서대로 환경에 적용 | 로봇·물체의 새 상태 |
| ⑥ 재관측과 종료 확인 | 각 환경 스텝에서 성공·종료를 확인하고, 계속할 경우 새 관측으로 다음 묶음 예측 | 새 영상·성공 여부·다음 행동 |

앞 10개를 실행하던 중에도 성공이나 종료 조건을 만나면 멈춥니다. 항상 10개를 끝까지 실행한 뒤에야 성공을 검사하는 것은 아닙니다. 10개를 정상 실행했다면 남은 40개를 그대로 잇지 않고 새 묶음을 얻습니다.

이 반복을 관리하고 파일을 남기는 것은 [교재 실행기](scripts/run_episode.py)입니다. **라이브러리가 제공하는 기능과, 교재가 정한 실행 순서를 구분해서 읽으세요.**


## 8. 뒤의 실습은 이 흐름을 세 단계로 나눕니다

| 실행 단계 | 어디까지 확인하는가 | 아직 확인하지 않은 것 |
|---|---|---|
| **inspect — 환경 관찰** | 환경 초기화, 지시·영상·상태 | 정책의 행동 예측과 작업 수행 |
| **infer — 행동 예측** | 초기 관측에서 정책을 실행하고 행동 묶음 확인 | 행동을 적용한 결과와 작업 성공 |
| **rollout — 반복 실행** | 행동 적용·재관측·새 예측을 반복 | 한 번의 성공만으로 다른 조건까지 보장하지는 못함 |

그래서 환경 사진이 보이면 `inspect` 단계는 진행된 것이지만, VLA가 작업을 수행했다는 뜻은 아닙니다. 50×7 행동 배열이 나왔다면 추론을 확인한 것이고, 실제 집기 결과는 `rollout`에서 봅니다.

`completed`는 실행 절차 완료, `success`는 과제 성공입니다. 환경 관찰만 수행한 결과는 정상 완료여도 아직 과제를 실행하지 않았습니다. 예외·사용자 중단은 별도로 기록합니다.

**같은 결과를 다른 단계의 성공으로 바꿔 읽지 않는 것** 이 뒤의 평가 실습을 이해하는 출발점입니다.


## 9. 영상이 20fps면 로봇도 실시간으로 20번 판단할까요?

그렇지 않습니다. **시뮬레이션의 시간, 계산에 걸린 실제 시간, 저장 영상의 재생 시간은 다릅니다.**

| 시간 또는 주기 | 뜻 |
|---|---|
| 물리 계산 스텝 | 물리 엔진이 상태를 적분하는 시간 간격 |
| 환경 제어 스텝 | 하나의 행동을 환경에 적용하는 단위. 내부 물리 계산이 여러 번 진행될 수 있음 |
| 정책 추론 시간 | 컴퓨터가 행동 묶음을 만드는 데 실제로 걸린 시간 |
| 저장 영상 fps | 저장한 프레임을 초당 몇 장으로 재생할지 정한 값 |

이번 실행기는 **정책을 계산하는 동안 다음 환경 스텝을 진행하지 않는 동기 방식** 입니다. 계산 후 행동을 적용하고, 갱신된 영상 프레임을 기록합니다.

예를 들어 저장된 프레임 100장을 20fps로 재생하면 영상 길이는 5초입니다. 이것만으로 해당 실행이 실제 5초 만에 끝났거나 정책이 초당 20번 판단했다고 말할 수 없습니다. 모델 준비·추론·파일 저장에는 별도의 시간이 걸릴 수 있습니다. 이 숫자는 구분을 설명하기 위한 예시입니다.

영상은 움직임을 보고, 기록된 시간은 계산 비용을 읽는 데 사용하세요. 실제 하드웨어의 반응성을 평가할 때에는 입력 도착과 행동 적용 시각도 확인해야 합니다.


## 10. 문제가 보이면 어느 부분부터 확인할까요?

아래는 원인을 확정하는 표가 아니라 **확인을 시작할 위치** 입니다. 같은 증상에도 여러 원인이 있을 수 있습니다.

| 관찰한 현상 | 먼저 확인할 부분 | 이유 |
|---|---|---|
| 가상 카메라 영상이 검게 나옴 | 렌더링 환경, 카메라 설정, 관측 전달 | 정책에 들어가기 전 영상이 이미 잘못됐을 수 있음 |
| 초기 그릇 배치가 예상과 다름 | 과제·초기 상태 선택 | 초기 배치와 seed를 혼동했을 수 있음 |
| 영상은 정상인데 행동 형상이 다름 | 체크포인트·입력·후처리 규격 | 모델과 연결 코드가 같은 규격을 사용하는지 확인 필요 |
| 로봇은 움직이지만 그릇을 못 잡음 | 입력 의미, 정책 행동, 초기 조건·접촉 결과 | 단순 연결 성공과 작업 수행 능력은 다름 |
| 잘 된 것처럼 보이지만 성공 값은 거짓 | 과제 지시와 환경 성공 조건 | 화면 인상과 정의된 성공 조건이 다를 수 있음 |
| 영상은 매끄럽지만 실행은 오래 걸림 | 준비 시간·추론 시간·영상 재생 설정 | 저장 fps는 계산 속도의 측정값이 아님 |

이 단계에서는 명령을 실행하지 않고 역할을 연결해 생각합니다. 실제 환경 점검 절차는 [03](03_container.ipynb)과 [복구 안내](README.md#복구)에서 진행합니다.


## 12. 다음 단계와 슬라이드 복습

이제 **“과제는 LIBERO, 물리·영상은 MuJoCo, 데이터·정책 연결은 LeRobot과 교재 코드, 행동 예측은 SmolVLA”** 라고 역할을 설명할 수 있어야 합니다. 이어서 관측→입력 변환→정책→후처리→환경 실행→새 관측 순서를 자신의 말로 풀어 보세요.

[03 컨테이너 준비](03_container.ipynb)에서 실행 환경을 확인한 뒤, [04 환경 관찰](04_environment.ipynb) → [05 관측·행동](05_observation_action.ipynb) → [06 추론](06_inference.ipynb) → [07 폐루프](07_rollout.ipynb) → [08 평가](08_evaluation.ipynb) 순서로 각 부분을 직접 확인합니다.
그림과 사례는 교재용 개념 설명이며 새로운 실행 측정 결과가 아닙니다. 모델의 기본 원리는 [01 VLA 이해하기](01_vla_theory.ipynb), 실제 입출력 규격은 [5일차 안내](README.md#모델과-데이터-계약)를 참고하세요.


## 실행 실습 — 설명용 mode=inspect / infer / rollout

세 모드의 호출 순서를 출력하고 env.step과 재관측이 rollout에서만 나타나는 것을 봅니다.

**구조 설명용 합성 예제입니다. GPU·모델·물리 엔진을 실행하지 않습니다.** 따라서 03 전용 환경 준비 전에도 기존 NumPy/Matplotlib 커널로 실행할 수 있습니다.


In [ ]:
# 실제 실행기의 세 모드를 단순화한 호출 순서 예제입니다. GPU와 시뮬레이터를 실행하지 않습니다.
def trace_mode(mode, execute=10):
    if mode not in ('inspect', 'infer', 'rollout'):
        raise ValueError('지원하지 않는 mode')
    events = ['LIBERO: task + initial state', 'MuJoCo: reset', 'adapter: make_batch', 'save: PNG + NPZ']
    if mode == 'inspect':
        return events
    events += ['SmolVLA: load_policy', 'LeRobot: preprocessor', 'policy: predict_action_chunk', 'postprocessor: action']
    if mode == 'rollout':
        events += [f'MuJoCo: env.step × 최대 {execute}', 'LIBERO: is_success 확인 / 매 step',
                   '새 관측 → 다음 predict_action_chunk (종료 전)']
    return events
for mode in ('inspect', 'infer', 'rollout'):
    print('\nMODE:', mode)
    for order, event in enumerate(trace_mode(mode), 1):
        print(f'{order:02d}. {event}')
assert not any('env.step' in e for e in trace_mode('infer'))
assert any('env.step' in e for e in trace_mode('rollout'))
print('\n정책 예측과 실제 적용의 경계를 확인했습니다. 실제 실행은 준비 검사 뒤 04부터 시작합니다.')


## 결과 해설과 다음 단계

시뮬레이터, 과제, 모델 연결, 정책의 역할을 분리하면 오류가 난 위치를 찾을 수 있습니다.

행동 묶음마다 reset하면 과제가 계속 처음으로 돌아갑니다. 추론 대기와 행동 적용을 구분하고 영상 fps를 벽시계 제어 주파수로 해석하지 않습니다.

이 흐름을 실제로 실행하기 전에 전용 Python·GPU·모델·Data 경로를 점검합니다.

출력 표·그래프·영상과 강사 설명을 함께 확인합니다. 별도 관찰 답안을 작성하지 않습니다.
